# CLAM-TR Phase 3 — TCGA-PRAD BENIGN Extraction (Q1 sprint follow-up to Phase 2)

**Purpose.** Replace the synthetic-95%-correct benign assumption used in the S118 audit (estimated QWK 0.71) with REAL model predictions on the 118 TCGA-PRAD Solid-Tissue-Normal slides. Output `predictions_20x_benign.csv` on Drive at `/content/drive/MyDrive/CLAM_TR_Results/TCGA_PRAD/`.

**Stage toggle.** `cfg['MAX_SLIDES'] = 3` for Stage 1 subset proof; change to `None` for Stage 2 full 118. Progress JSON is benign-specific so Stage 2 resumes cleanly from Stage 1.

**This notebook is derived from `T2_1_phase2_tcga_prad_external_validation_colab.ipynb` by `.claude/bin/build_phase3_benign_notebook.py`. Do not edit the cells inline; regenerate via the build script if you need changes.**

## 1. Setup & Imports

In [ ]:
# Install dependencies + mount Drive (run once per Colab session)
!pip install -q openslide-python opencv-python-headless timm huggingface_hub requests
!apt-get install -y openslide-tools aria2 >/dev/null 2>&1

from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, shutil, queue, threading
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F

import openslide
import cv2
from PIL import Image
import timm
from huggingface_hub import hf_hub_download, login as hf_login
from sklearn.metrics import cohen_kappa_score, confusion_matrix, accuracy_score
from tqdm import tqdm

# ── HuggingFace Auth (needed for MahmoodLab/UNI gated model) ──
# Token from the project's HF account (matches Phase 1 setup cell).
# Get your token from: https://huggingface.co/settings/tokens
# Also request access at: https://huggingface.co/MahmoodLab/UNI
HF_TOKEN = "hf_PLACEHOLDER_INSERT_YOUR_HF_TOKEN_HERE"
hf_login(token=HF_TOKEN)
print("HuggingFace: Logged in")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')


## 2. Configuration

Default paths assume a standard Drive layout. Only edit if your v20 checkpoints live elsewhere.

In [ ]:
# ============================================================
# DEFAULT PATHS (edit V20_CHECKPOINT_DIR if needed)
# ============================================================

TCGA_ROOT          = '/content/drive/MyDrive/TCGA_PRAD/'
TCGA_FEATURES_DIR  = '/content/drive/MyDrive/CLAM_TR_Features/TCGA_PRAD/uni_20x_benign/'
TCGA_PROGRESS_DIR  = '/content/drive/MyDrive/CLAM_TR_Preprocessing/TCGA_PRAD/progress_20x/'

# Auto-fetched files (you do NOT need to download these manually)
GDC_MANIFEST       = os.path.join(TCGA_ROOT, 'gdc_manifest.tsv')
TCGA_CLINICAL_CSV  = os.path.join(TCGA_ROOT, 'clinical.csv')
TCGA_LABELS_CSV    = os.path.join(TCGA_ROOT, 'labels_benign.csv')

# Local Colab temp dir (slides streamed here then deleted after extraction)
LOCAL_SLIDES_DIR   = '/content/tcga_local_20x/'

# *** ONLY EDIT IF YOUR V20 CHECKPOINTS LIVE ELSEWHERE ***
# Default = the v20 RetNet_bestHP-on-UNI 5-fold checkpoints written by
# CLAM_TR_v20_UNI_Colab.ipynb (CheckpointManager layout: BASE_RESULTS_DIR/
# v20_fullscale/UNI/checkpoints/{exp_name}_fold{N}_epoch.pt, containing
# 'best_model_state_dict' = early-stopping-selected best-QWK weights).
V20_CHECKPOINT_DIR = '/content/drive/MyDrive/CLAM_TR_Results/v20_fullscale/UNI/checkpoints/'
V20_EXP_NAME       = 'RetNet_bestHP'   # filename prefix written by v20 trainer
# Expected files: RetNet_bestHP_fold0_epoch.pt ... RetNet_bestHP_fold4_epoch.pt
# (legacy fallback names: fold_N.pt, fold_N_best.pt, best_fold_N.pt, v20_fold_N.pt)

# Output paths
TCGA_PREDICTIONS_CSV = '/content/drive/MyDrive/CLAM_TR_Results/TCGA_PRAD/predictions_20x_benign.csv'
TCGA_METRICS_JSON    = '/content/drive/MyDrive/CLAM_TR_Results/TCGA_PRAD/metrics_20x_benign.json'

# ============================================================
# Pipeline settings — MUST MATCH Phase 1 for feature compatibility
# ============================================================
# Auto-set GPU batch size based on available VRAM
# (A100 40GB -> 512; T4 15GB -> 256)
_vram_gb = (torch.cuda.get_device_properties(0).total_memory / 1e9
            if torch.cuda.is_available() else 0)
_default_batch = 512 if _vram_gb > 25 else 256
print(f'Detected VRAM: {_vram_gb:.1f} GB -> encoder_batch_size = {_default_batch}')

EXTRACTION_CONFIG = {
    'patch_size':              256,
    'target_magnification':    20,        # 20x — matches v20 PANDA UNI training (effective magnification of /content/drive/MyDrive/UNI_features/)
    'encoder_batch_size':      _default_batch,   # 512 on A100, 256 on T4
    'checkpoint_every':        10,
    'min_tissue_ratio':        0.5,
    'sat_thresh':              8,
    'median_blur':             7,
    'morph_close':             4,
    'white_sat_thresh':        5,
    'black_rgb_thresh':        40,
    'min_patches_per_slide':   10,
    # ---- Subset cap (None = full 426-slide cohort). Was 30 for the diagnostic proof. ----
    'MAX_SLIDES':              3,         # STAGE 1 subset proof (3 benign slides). Change to None for STAGE 2 full 118 cohort.
    'n_workers':               4,         # 20x extraction: 8 workers OOM Colab's 53 GB ceiling (~62 GB peak); 2 workers were CPU-bound (~7 min/slide because TCGA SVS downloads from GDC are 3-7 min each). 4 workers balance the pipeline: ~36 GB peak RAM, ~3-4 min/slide throughput.
    'aria2c_connections':      4,         # parallel TCP streams per single slide via aria2c
}

# Model + inference settings (must match v20 RetNet_bestHP)
MODEL_CONFIG = {
    'input_dim':              1024,
    'hidden_dim':             512,
    'n_classes':              6,
    'num_heads':              8,
    'K':                      128,
    'ffn_expansion':          4,
    'num_groups':             8,
    'gamma_bank':             [0.88, 0.90, 0.92, 0.94, 0.96, 0.98, 0.99, 0.995],
    'attention_temperature':  0.7,
    'dropout':                0.15,
    'use_spatial':            False,
    'use_norm':               True,
    'use_residual':           True,
    'use_gate':               True,
    'use_ffn':                True,
}

# Bootstrap CI for QWK
N_BOOTSTRAP = 1000
RNG_SEED    = 42

# Create output directories
for d in [TCGA_ROOT, TCGA_FEATURES_DIR, TCGA_PROGRESS_DIR, LOCAL_SLIDES_DIR,
          os.path.dirname(TCGA_PREDICTIONS_CSV), os.path.dirname(TCGA_METRICS_JSON)]:
    os.makedirs(d, exist_ok=True)

print('Configuration applied.')
print(f'  TCGA root:         {TCGA_ROOT}')
print(f'  Features dir:      {TCGA_FEATURES_DIR}')
print(f'  Checkpoint dir:    {V20_CHECKPOINT_DIR}')
print(f'  Local stream dir:  {LOCAL_SLIDES_DIR}')
print(f'  Predictions CSV:   {TCGA_PREDICTIONS_CSV}')
print(f'  Metrics JSON:      {TCGA_METRICS_JSON}')


## 3. (SKIPPED) Auto-fetch GDC Manifest + Clinical Metadata

**Benign extraction does NOT use this cell.** Cell 4 below builds `labels_benign.csv` directly from a GDC REST query for the 118 TCGA-PRAD Solid-Tissue-Normal slides (all ISUP=0). The cancer-side GDC manifest + clinical metadata are irrelevant for benign extraction.

_Original Phase 2 cell content removed; placeholder cells below print a marker and return._

In [ ]:
print('Cell 3 SKIPPED for benign extraction. See Cell 4 (benign manifest builder).')


In [ ]:
print('Cell 3 SKIPPED for benign extraction. See Cell 4 (benign manifest builder).')


## 4. Build labels_benign.csv from GDC

Queries the GDC REST API for all 118 TCGA-PRAD Solid-Tissue-Normal SVS files (sample_type=11, all open access), writes `labels_benign.csv` to `TCGA_LABELS_CSV`. All 118 benign slides get `isup_grade=0` by definition.

In [ ]:
# Build labels_benign.csv directly from a GDC REST query (no manual portal browsing).
import os, json, urllib.request, urllib.parse
import pandas as pd

_filters = {
    'op': 'and',
    'content': [
        {'op': 'in', 'content': {'field': 'cases.project.project_id', 'value': ['TCGA-PRAD']}},
        {'op': 'in', 'content': {'field': 'cases.samples.sample_type', 'value': ['Solid Tissue Normal']}},
        {'op': 'in', 'content': {'field': 'data_format', 'value': ['SVS']}},
    ],
}
_params = {
    'filters': json.dumps(_filters),
    'size': '500',
    'fields': 'file_id,file_name,file_size,cases.submitter_id,cases.samples.sample_type',
    'format': 'JSON',
}
_url = 'https://api.gdc.cancer.gov/files?' + urllib.parse.urlencode(_params)
_req = urllib.request.Request(_url, headers={'User-Agent': 'clam-tr-q1-sprint/1.0'})
with urllib.request.urlopen(_req, timeout=60) as _r:
    _data = json.loads(_r.read().decode('utf-8'))

_hits = _data['data']['hits']
print(f'GDC returned {len(_hits)} TCGA-PRAD Solid-Tissue-Normal SVS files (open access)')

_rows = []
for _h in _hits:
    _fname = _h['file_name']
    _slide_id = _fname[:-4] if _fname.lower().endswith('.svs') else _fname
    _rows.append({
        'slide_id':          _slide_id,
        'gdc_file_uuid':     _h['file_id'],
        'file_size_bytes':   _h.get('file_size', 0),
        'isup_grade':        0,
        'gleason_primary':   0,
        'gleason_secondary': 0,
    })

_rows.sort(key=lambda r: r['slide_id'])  # deterministic ordering
labels_df = pd.DataFrame(_rows)
os.makedirs(os.path.dirname(TCGA_LABELS_CSV), exist_ok=True)
labels_df.to_csv(TCGA_LABELS_CSV, index=False)

print(f'Wrote {TCGA_LABELS_CSV}: {len(labels_df)} benign slides')
print(f'  total payload: {labels_df["file_size_bytes"].sum()/1e9:.2f} GB')
print(f'  size stats (MB): min={labels_df["file_size_bytes"].min()/1e6:.1f}'
      f'  median={labels_df["file_size_bytes"].median()/1e6:.1f}'
      f'  max={labels_df["file_size_bytes"].max()/1e6:.1f}')
print()
print(f'First 5 slide_ids:')
for sid in labels_df['slide_id'].head(5):
    print(f'  {sid}')
print()
print(f'Ready for Cell 5 (run_streaming_extraction). '
      f'EXTRACTION_CONFIG["MAX_SLIDES"] = {EXTRACTION_CONFIG.get("MAX_SLIDES")}')


## 5. Feature Extraction Pipeline (streaming)

Reuses Phase 1's `SlidePreprocessor` + UNI v1 encoder so the TCGA-PRAD features lie in the same representation space as the PANDA training features. The streaming loop downloads each slide to local Colab disk, extracts features, saves the `.pt` to Drive, then deletes the local `.svs` — Drive footprint stays <1 GB.

In [ ]:
# Slide Preprocessor — IDENTICAL to Phase 1
class SlidePreprocessor:
    def __init__(self, patch_size=256, target_mag=5, min_tissue_ratio=0.5,
                 sat_thresh=8, median_blur=7, morph_close=4,
                 white_sat_thresh=5, black_rgb_thresh=40):
        self.patch_size = patch_size
        self.target_mag = target_mag
        self.min_tissue_ratio = min_tissue_ratio
        self.sat_thresh = sat_thresh
        self.median_blur = median_blur
        self.morph_close = morph_close
        self.white_sat_thresh = white_sat_thresh
        self.black_rgb_thresh = black_rgb_thresh

    def get_extraction_params(self, slide):
        try:
            obj_power = float(slide.properties.get('openslide.objective-power', 20))
        except (ValueError, KeyError):
            obj_power = 20
        downsample_needed = obj_power / self.target_mag
        best_level, best_diff = 0, float('inf')
        for level in range(slide.level_count):
            diff = abs(slide.level_downsamples[level] - downsample_needed)
            if diff < best_diff:
                best_diff, best_level = diff, level
        level_ds = slide.level_downsamples[best_level]
        actual_mag = obj_power / level_ds
        mag_ratio = actual_mag / self.target_mag
        read_size = int(round(self.patch_size * mag_ratio))
        step_l0   = int(round(read_size * level_ds))
        return best_level, read_size, step_l0, actual_mag, obj_power

    def segment_tissue(self, slide):
        seg_level = 0
        for level in range(slide.level_count):
            if slide.level_downsamples[level] <= 64:
                seg_level = level
        dims = slide.level_dimensions[seg_level]
        img = np.array(slide.read_region((0, 0), seg_level, dims).convert('RGB'))
        img_hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        img_med = cv2.medianBlur(img_hsv[:, :, 1], self.median_blur)
        _, tissue_mask = cv2.threshold(img_med, self.sat_thresh, 255, cv2.THRESH_BINARY)
        if self.morph_close > 0:
            kernel = np.ones((self.morph_close, self.morph_close), np.uint8)
            tissue_mask = cv2.morphologyEx(tissue_mask, cv2.MORPH_CLOSE, kernel)
        return tissue_mask, seg_level

    def extract_patches_and_coords(self, slide):
        level, read_size, step_l0, actual_mag, obj_power = self.get_extraction_params(slide)
        tissue_mask, seg_level = self.segment_tissue(slide)
        seg_ds = slide.level_downsamples[seg_level]
        mask_h, mask_w = tissue_mask.shape
        needs_resize = (read_size != self.patch_size)
        patches, coords = [], []
        w0, h0 = slide.level_dimensions[0]
        for y in range(0, h0 - step_l0 + 1, step_l0):
            for x in range(0, w0 - step_l0 + 1, step_l0):
                mx = min(int(x / seg_ds), mask_w - 1)
                my = min(int(y / seg_ds), mask_h - 1)
                mw = max(1, min(int(step_l0 / seg_ds), mask_w - mx))
                mh = max(1, min(int(step_l0 / seg_ds), mask_h - my))
                if mw <= 0 or mh <= 0:
                    continue
                tissue_ratio = np.mean(tissue_mask[my:my+mh, mx:mx+mw] > 0)
                if tissue_ratio < self.min_tissue_ratio:
                    continue
                try:
                    patch_pil = slide.read_region((x, y), level, (read_size, read_size)).convert('RGB')
                    patch_arr = np.array(patch_pil)
                    patch_hsv = cv2.cvtColor(patch_arr, cv2.COLOR_RGB2HSV)
                    if np.mean(patch_hsv[:, :, 1]) < self.white_sat_thresh:
                        continue
                    if np.all(np.mean(patch_arr, axis=(0, 1)) < self.black_rgb_thresh):
                        continue
                    if needs_resize:
                        patch_pil = patch_pil.resize((self.patch_size, self.patch_size), Image.LANCZOS)
                        patch_arr = np.array(patch_pil)
                    patches.append(patch_arr)
                    coords.append([x, y])
                except Exception:
                    continue
        return patches, np.array(coords, dtype=np.int32) if coords else np.zeros((0, 2), dtype=np.int32)


class UNIv1Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        print('Loading UNI v1 (ViT-L/16)...')
        self.encoder = timm.create_model('vit_large_patch16_224', img_size=224, patch_size=16,
                                          init_values=1e-5, num_classes=0, dynamic_img_size=True)
        # MahmoodLab/UNI is GATED; auth already handled via hf_login() at cell 2 startup.
        weights_path = hf_hub_download('MahmoodLab/UNI', 'pytorch_model.bin')
        self.encoder.load_state_dict(torch.load(weights_path, map_location='cpu', weights_only=False), strict=True)
        for p in self.encoder.parameters():
            p.requires_grad = False
        print(f'UNI v1 loaded: {sum(p.numel() for p in self.parameters()):,} params (frozen)')

    def forward(self, x):
        return self.encoder(x)

print('Pipeline classes defined.')


## 6. Streaming Feature Extraction (download → extract → delete)

Loops over every TCGA-PRAD slide in the manifest. For each one:
1. Stream-download via GDC data endpoint (`https://api.gdc.cancer.gov/data/{uuid}`) to local Colab disk.
2. Open with OpenSlide + extract patches via the IDENTICAL Phase 1 preprocessing.
3. Encode patches with UNI v1 (frozen) → `.pt` features on Drive.
4. Delete the local `.svs`.

**Resume-safe**: re-run the cell after a Colab disconnect; it picks up from `progress.json`.

In [ ]:
GDC_DATA_URL = 'https://api.gdc.cancer.gov/data'
PROGRESS_PATH = os.path.join(TCGA_PROGRESS_DIR, 'uni_extraction_progress_benign.json')


import subprocess
import shutil as _shutil

_ARIA2C_AVAILABLE = _shutil.which('aria2c') is not None
if _ARIA2C_AVAILABLE:
    print(f'aria2c detected at {_shutil.which("aria2c")} -- using multi-connection downloads')
else:
    print('aria2c NOT detected -- falling back to single-connection requests downloads')


def stream_download_slide(file_uuid, file_name, dest_dir, chunk_size=8 * 1024 * 1024):
    """Stream a single slide from GDC data endpoint to local disk.

    Uses aria2c with N parallel TCP streams per file (~3-4x faster than single-stream
    requests). Falls back to requests if aria2c not installed.

    Returns the local path on success, None on failure."""
    dest_path = os.path.join(dest_dir, file_name)
    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 0:
        return dest_path
    url = f'{GDC_DATA_URL}/{file_uuid}'

    # Path A: aria2c multi-connection
    if _ARIA2C_AVAILABLE:
        n_conn = EXTRACTION_CONFIG.get('aria2c_connections', 4)
        try:
            result = subprocess.run(
                ['aria2c',
                 '-x', str(n_conn),               # max connections per server
                 '-s', str(n_conn),               # split (parallel streams per file)
                 '--max-tries=3',
                 '--retry-wait=5',
                 '--summary-interval=0',
                 '--console-log-level=warn',
                 '--allow-overwrite=true',
                 '-o', file_name,
                 '-d', dest_dir,
                 url],
                capture_output=True, timeout=600,
            )
            if result.returncode == 0 and os.path.exists(dest_path) and os.path.getsize(dest_path) > 0:
                return dest_path
            # Fall through to requests on aria2c failure
        except (subprocess.TimeoutExpired, FileNotFoundError):
            pass

    # Path B: single-connection requests fallback
    try:
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(dest_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=chunk_size):
                    if chunk:
                        f.write(chunk)
        return dest_path
    except Exception as e:
        print(f'  DOWNLOAD FAILED {file_uuid[:8]}...: {e}')
        if os.path.exists(dest_path):
            try:
                os.remove(dest_path)
            except Exception:
                pass
        return None


def run_streaming_extraction():
    """Multi-worker streaming extraction.

    Mirrors Phase 1's multi-worker pipeline: N CPU workers download slides from GDC
    + extract patches in parallel; 1 GPU main thread does UNI inference + saves .pt
    + deletes local. Drive footprint stays under 1 GB.

    Resume-safe via progress JSON. Per-slide local download deleted after encode.
    """
    cfg = EXTRACTION_CONFIG
    N_WORKERS = cfg.get('n_workers', 5)

    # Load progress
    if os.path.exists(PROGRESS_PATH):
        with open(PROGRESS_PATH) as f:
            progress = json.load(f)
        completed_prev       = set(progress.get('completed', []))
        failed_prev          = set(progress.get('failed', []))
        insufficient_prev    = set(progress.get('insufficient_tissue', []))
    else:
        completed_prev, failed_prev, insufficient_prev = set(), set(), set()

    labels_df = pd.read_csv(TCGA_LABELS_CSV)

    # ---- Subset cap (for the 20x diagnostic proof) ----
    _max = cfg.get('MAX_SLIDES', None)
    if _max and _max > 0 and len(labels_df) > _max:
        _n_grades = labels_df['isup_grade'].nunique()
        _per_grade = max(1, _max // _n_grades)
        # Deterministic stratified sample: first N slides per ISUP grade (sorted by slide_id)
        labels_df = (labels_df.sort_values('slide_id')
                              .groupby('isup_grade', group_keys=False, sort=True)
                              .head(_per_grade)
                              .reset_index(drop=True))
        print(f'[SUBSET MODE] Capped to {len(labels_df)} slides '
              f'({_per_grade}/grade x {_n_grades} grades, stratified by ISUP)')
        print(labels_df['isup_grade'].value_counts().sort_index().to_string())
    # ---- end subset cap ----

    # Re-scan existing .pt files
    existing_pts = set()
    if os.path.isdir(TCGA_FEATURES_DIR):
        existing_pts = {f.replace('.pt', '') for f in os.listdir(TCGA_FEATURES_DIR) if f.endswith('.pt')}
    completed_prev = completed_prev | existing_pts

    # Slides to process: in labels CSV, not yet completed, not permanently skipped
    to_process_df = labels_df[~labels_df['slide_id'].isin(completed_prev | insufficient_prev)]

    print('=' * 60)
    print('TCGA-PRAD MULTI-WORKER STREAMING EXTRACTION')
    print('=' * 60)
    print(f'Total slides with labels:                 {len(labels_df)}')
    print(f'Already done:                             {len(completed_prev)}')
    print(f'Permanently skipped (insufficient tissue): {len(insufficient_prev)}')
    print(f'To process this session:                  {len(to_process_df)}')
    print(f'CPU workers:                              {N_WORKERS}')

    if len(to_process_df) == 0:
        print('\nAll slides processed. Extraction complete.')
        return

    # Load encoder ONCE in main thread (shared, inference-only)
    encoder = UNIv1Encoder().to(device).eval()
    _gpu_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    _gpu_std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    preprocessor = SlidePreprocessor(
        patch_size=cfg['patch_size'], target_mag=cfg['target_magnification'],
        min_tissue_ratio=cfg['min_tissue_ratio'], sat_thresh=cfg.get('sat_thresh', 8),
        median_blur=cfg.get('median_blur', 7), morph_close=cfg.get('morph_close', 4),
        white_sat_thresh=cfg.get('white_sat_thresh', 5),
        black_rgb_thresh=cfg.get('black_rgb_thresh', 40))

    # ── Queues + shared state ──
    slide_ids_q = queue.Queue()
    for _, row in to_process_df.iterrows():
        slide_ids_q.put((row['slide_id'], row['gdc_file_uuid']))
    gpu_q = queue.Queue(maxsize=N_WORKERS)
    completed_list = list(completed_prev)
    failed_list = list(failed_prev)
    insufficient_tissue = set(insufficient_prev)
    preprocess_errors = []
    _lock = threading.Lock()

    def _cpu_worker():
        """Download slide from GDC -> extract patches as raw numpy, push to GPU queue."""
        while True:
            try:
                slide_id, file_uuid = slide_ids_q.get_nowait()
            except Exception:
                return

            file_name = f'{slide_id}.svs'
            # 1. Download to local Colab disk
            local_path = stream_download_slide(file_uuid, file_name, LOCAL_SLIDES_DIR)
            if local_path is None:
                gpu_q.put((slide_id, None, None, None, 'download_failed'))
                continue

            # 2. Open + extract patches
            try:
                slide = openslide.OpenSlide(local_path)
                patches, coords = preprocessor.extract_patches_and_coords(slide)
                slide.close()
            except Exception as e:
                with _lock:
                    preprocess_errors.append((slide_id, f'openslide: {e}'))
                gpu_q.put((slide_id, None, None, local_path, 'openslide_error'))
                continue

            # 3. Tissue-floor reject?
            if len(patches) < cfg['min_patches_per_slide']:
                with _lock:
                    insufficient_tissue.add(slide_id)
                gpu_q.put((slide_id, None, None, local_path, 'insufficient_tissue'))
                continue

            # 4. Stack patches -> push to GPU queue
            try:
                patch_tensors = torch.from_numpy(np.stack(patches))
                coords_tensor = torch.tensor(coords, dtype=torch.float32)
                del patches
                gpu_q.put((slide_id, patch_tensors, coords_tensor, local_path, None))
            except Exception as e:
                with _lock:
                    preprocess_errors.append((slide_id, f'stack: {e}'))
                gpu_q.put((slide_id, None, None, local_path, 'stack_error'))

    # ── Start CPU workers ──
    actual_workers = min(N_WORKERS, len(to_process_df))
    cpu_threads = []
    for _ in range(actual_workers):
        t = threading.Thread(target=_cpu_worker, daemon=True)
        t.start()
        cpu_threads.append(t)
    print(f'  {actual_workers} CPU workers started')

    # ── GPU loop (main thread) ──
    batch_size = cfg['encoder_batch_size']
    session_count = 0
    processed_count = 0
    total = len(to_process_df)
    start_time = time.time()
    pbar = tqdm(total=total, desc='Streaming', unit='slide')

    while processed_count < total:
        try:
            item = gpu_q.get(timeout=600)   # 10-min timeout per slide (large .svs downloads)
        except Exception:
            alive = sum(1 for t in cpu_threads if t.is_alive())
            print(f'\n  GPU queue timeout. {alive}/{actual_workers} CPU workers still alive')
            if alive == 0:
                break
            continue

        slide_id, patch_tensors, coords_tensor, local_path, err = item
        processed_count += 1

        # Cleanup local file regardless of outcome
        def _cleanup():
            if local_path and os.path.exists(local_path):
                try: os.remove(local_path)
                except Exception: pass

        if patch_tensors is None:
            # Error or tissue-floor reject path
            if err == 'download_failed':
                failed_list.append(slide_id)
            elif err == 'insufficient_tissue':
                pass   # already in insufficient_tissue set
            else:
                failed_list.append(slide_id)
            _cleanup()
            pbar.update(1)
            continue

        # GPU encode
        try:
            all_features = []
            with torch.inference_mode(), torch.amp.autocast('cuda'):
                for b in range(0, len(patch_tensors), batch_size):
                    batch_raw = patch_tensors[b:b + batch_size]
                    batch_gpu = batch_raw.to(device).permute(0, 3, 1, 2).float() / 255.0
                    batch_gpu = F.interpolate(batch_gpu, size=(224, 224), mode='bilinear', align_corners=False)
                    batch_gpu = (batch_gpu - _gpu_mean) / _gpu_std
                    feats = encoder(batch_gpu).float().cpu()
                    all_features.append(feats)
                    del batch_raw, batch_gpu
            features_cat = torch.cat(all_features, dim=0)

            # Save .pt to Drive
            out_path = os.path.join(TCGA_FEATURES_DIR, f'{slide_id}.pt')
            torch.save({'features': features_cat, 'coords': coords_tensor}, out_path)
            completed_list.append(slide_id)
            session_count += 1
            del patch_tensors, all_features, features_cat, coords_tensor
        except Exception as e:
            print(f'\n  GPU ERROR {slide_id}: {e}')
            failed_list.append(slide_id)

        _cleanup()
        pbar.update(1)

        # Progress bar postfix + checkpoint
        elapsed = time.time() - start_time
        if session_count:
            rate_per_min = session_count / (elapsed / 60)
            eta_min = (total - processed_count) / max(rate_per_min, 0.01)
            pbar.set_postfix(rate=f'{rate_per_min:.1f}/min', eta=f'{eta_min:.0f}min')

        if session_count and session_count % cfg['checkpoint_every'] == 0:
            with open(PROGRESS_PATH, 'w') as f:
                json.dump({
                    'completed':            sorted(set(completed_list)),
                    'failed':               sorted(set(failed_list)),
                    'insufficient_tissue':  sorted(insufficient_tissue),
                    'total':                len(labels_df),
                    'timestamp':            datetime.now().isoformat(),
                }, f)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    pbar.close()

    # Wait for CPU workers
    for t in cpu_threads:
        t.join(timeout=10)

    # Final save
    with open(PROGRESS_PATH, 'w') as f:
        json.dump({
            'completed':            sorted(set(completed_list)),
            'failed':               sorted(set(failed_list)),
            'insufficient_tissue':  sorted(insufficient_tissue),
            'total':                len(labels_df),
            'timestamp':            datetime.now().isoformat(),
        }, f)

    elapsed = time.time() - start_time
    new_insufficient = len(insufficient_tissue) - len(insufficient_prev)
    print(f'\n=== EXTRACTION COMPLETE ===')
    print(f'  Session: {session_count} new slides in {elapsed/60:.1f} min  ({elapsed/max(1,session_count):.1f} sec/slide avg)')
    print(f'  Total: {len(set(completed_list))}/{len(labels_df)} done')
    print(f'  Failed (transient):                        {len(set(failed_list))}')
    print(f'  Insufficient tissue (permanent skip):      {len(insufficient_tissue)} total (+{new_insufficient} this session)')
    if preprocess_errors:
        print(f'  Genuine preprocess errors (first 5):')
        for sid, err in preprocess_errors[:5]:
            print(f'    - {sid}: {err}')
    print(f'  Output dir: {TCGA_FEATURES_DIR}')

    shutil.rmtree(LOCAL_SLIDES_DIR, ignore_errors=True)
    os.makedirs(LOCAL_SLIDES_DIR, exist_ok=True)
    del encoder
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


run_streaming_extraction()


## 7. Model Architecture — CLAM-TR-RetNet (v20 RetNet_bestHP)

Provenance: `CLAM_TR_v20_*.ipynb` cells 21 + 23. If `load_state_dict()` fails with key mismatch, inspect the checkpoint structure:
```python
ckpt = torch.load('fold_0.pt', weights_only=False)
print(list(ckpt.keys())[:20])
```
and adjust the class hierarchy below to match (the v20 notebook is the authoritative reference).

In [ ]:
# ============================================================
# v20 V7-exact model classes (verbatim from CLAM_TR_v20_UNI_Colab.ipynb)
# Required for state_dict compatibility with the trained
# RetNet_bestHP_fold{N}_epoch.pt checkpoints.
# ============================================================

class GatedAttention(nn.Module):
    """V7 exact. Gated attention with temperature scaling."""
    def __init__(self, input_dim=512, hidden_dim=256, dropout=0.25):
        super().__init__()
        self.attention_a = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Tanh(), nn.Dropout(dropout))
        self.attention_b = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Sigmoid(), nn.Dropout(dropout))
        self.attention_c = nn.Linear(hidden_dim, 1)

    def forward(self, x, temperature=1.0):
        a = self.attention_a(x)
        b = self.attention_b(x)
        return self.attention_c(a * b).squeeze(-1) / temperature


class SwishGate(nn.Module):
    """V7 exact."""
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        return x * torch.sigmoid(self.proj(x))


class FFNBlock(nn.Module):
    """V7 exact. Pre-norm residual FFN."""
    def __init__(self, dim, expansion=4, dropout=0.1):
        super().__init__()
        hidden = dim * expansion
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim), nn.Dropout(dropout))
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return x + self.net(self.norm(x))


class RetentionHead(nn.Module):
    """V7 exact. Scaled dot-product attention (no spatial decay).

    NOTE: in v20 the 'RetNet' variant is plain multi-head scaled-dot-product
    attention with softmax -- NOT the retentive-network causal-decay
    formulation. The 'RetNet' name is historical; the actual head computes:
        scores = Q K^T * head_dim^-0.5
        attn   = softmax(scores)
        out    = attn @ V
    """
    def __init__(self, hidden_dim, num_heads):
        super().__init__()
        self.head_dim = hidden_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.W_Q = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_K = nn.Linear(hidden_dim, self.head_dim, bias=False)
        self.W_V = nn.Linear(hidden_dim, self.head_dim, bias=False)

    def forward(self, x):
        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn = F.softmax(scores, dim=-1)
        return torch.matmul(attn, V)


class RetentionBlock(nn.Module):
    """V7 RetentionEnhanced exact. RetNet_bestHP uses use_spatial=False.
    Order: MHA -> output_proj -> dropout -> [Norm -> Residual -> Gate -> FFN]"""

    def __init__(self, hidden_dim=512, num_heads=8, gamma_values=None,
                 use_spatial=False, spatial_mode='multiplicative',
                 use_norm=True, norm_type='group', num_groups=8,
                 use_residual=True, use_gate=True, use_ffn=True,
                 ffn_expansion=4, dropout=0.25):
        super().__init__()
        self.use_spatial = use_spatial
        self.use_norm = use_norm
        self.use_residual = use_residual
        self.use_gate = use_gate
        self.use_ffn = use_ffn

        # RetNet_bestHP path: plain RetentionHead per head (no spatial decay)
        self.heads = nn.ModuleList([
            RetentionHead(hidden_dim, num_heads) for _ in range(num_heads)])

        self.output_proj = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)

        if use_norm:
            if norm_type == 'group':
                self.norm = nn.GroupNorm(num_groups, hidden_dim)
            else:
                self.norm = nn.LayerNorm(hidden_dim)
        if use_gate:
            self.gate = SwishGate(hidden_dim)
        if use_ffn:
            self.ffn = FFNBlock(hidden_dim, ffn_expansion, dropout)

    def forward(self, x, coords=None):
        identity = x
        head_outputs = [head(x) for head in self.heads]
        out = self.output_proj(torch.cat(head_outputs, dim=-1))
        out = self.dropout(out)
        # V7 EXACT ORDER: Norm -> Residual -> Gate -> FFN
        if self.use_norm:
            out = self.norm(out)
        if self.use_residual:
            out = out + identity
        if self.use_gate:
            out = self.gate(out)
        if self.use_ffn:
            out = self.ffn(out)
        return out


class DualBranchModel(nn.Module):
    """v20 V7-exact dual-branch model with TopK selection and addition fusion."""
    def __init__(self, input_dim, hidden_dim=512, n_classes=6, dropout=0.25,
                 K=128, retention_module=None, needs_coords=False):
        super().__init__()
        self.K = K
        self.needs_coords = needs_coords
        self.feature_transform = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout))
        self.attention = GatedAttention(hidden_dim, hidden_dim // 2, dropout)
        self.retention_module = retention_module
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, x, coords=None, temperature=1.0):
        h = self.feature_transform(x)

        # Attention scores (raw, before softmax)
        A = self.attention(h, temperature)
        A_softmax = F.softmax(A, dim=0)

        # TopK selection by RAW attention
        K_actual = min(self.K, h.size(0))
        topk_indices = torch.topk(A, K_actual).indices
        h_topk = h[topk_indices]
        A_topk = A_softmax[topk_indices]

        # CLAM branch: TopK weighted sum (V7 exact, NOT re-normalized)
        M_clam = torch.sum(A_topk.unsqueeze(-1) * h_topk, dim=0)

        # Retention branch
        if self.retention_module is None:
            M_retention = h_topk.mean(dim=0)
        elif self.needs_coords:
            coords_topk = coords[topk_indices]
            M_retention = self.retention_module(h_topk, coords_topk).mean(dim=0)
        else:
            M_retention = self.retention_module(h_topk).mean(dim=0)

        # Addition fusion (V7 exact)
        return self.classifier(M_clam + M_retention)


def make_retnet_besthp_model(model_config):
    """Build a RetNet_bestHP-on-UNI model exactly as v20 create_model() does."""
    retention = RetentionBlock(
        hidden_dim=model_config['hidden_dim'],
        num_heads=model_config['num_heads'],
        use_spatial=False,
        use_norm=model_config.get('use_norm', True),
        norm_type='group',
        num_groups=model_config['num_groups'],
        use_residual=model_config.get('use_residual', True),
        use_gate=model_config.get('use_gate', True),
        use_ffn=model_config.get('use_ffn', True),
        ffn_expansion=model_config['ffn_expansion'],
        dropout=model_config['dropout'],
    )
    return DualBranchModel(
        input_dim=model_config['input_dim'],
        hidden_dim=model_config['hidden_dim'],
        n_classes=model_config['n_classes'],
        dropout=model_config['dropout'],
        K=model_config['K'],
        retention_module=retention,
        needs_coords=False,
    )


# Sanity check: parameter count should match the trained checkpoint (4,204,295)
_probe = make_retnet_besthp_model(MODEL_CONFIG)
_n_params = sum(p.numel() for p in _probe.parameters())
print(f'Model architecture defined. Param count: {_n_params:,} (expected 4,204,295 for v20 RetNet_bestHP)')
assert _n_params == 4204295, f'Param-count mismatch: {_n_params} != 4,204,295 -- architecture diverges from v20'
del _probe


## 8. Load 5-Fold Ensemble + Run Inference + Evaluate

Loads the v20 RetNet_bestHP checkpoints, averages softmax across 5 folds per slide, computes QWK + per-grade accuracy + bootstrap 95% CI, and saves predictions + metrics.

In [ ]:
# ============================================================
# Load v20 5-fold ensemble
# ============================================================
fold_models = {}
_EXP = globals().get('V20_EXP_NAME', 'RetNet_bestHP')
for fold in range(5):
    # Search order: native v20-Colab layout first, then legacy fallbacks
    candidates = [
        f'{_EXP}_fold{fold}_epoch.pt',    # v20-Colab CheckpointManager (native)
        f'fold_{fold}.pt',
        f'fold_{fold}_best.pt',
        f'best_fold_{fold}.pt',
        f'v20_fold_{fold}.pt',
    ]
    ckpt_path = None
    for name in candidates:
        p = os.path.join(V20_CHECKPOINT_DIR, name)
        if os.path.exists(p):
            ckpt_path = p
            break
    if ckpt_path is None:
        raise FileNotFoundError(
            f'Checkpoint not found for fold {fold} in {V20_CHECKPOINT_DIR}\n'
            f'Expected one of: ' + ', '.join(candidates) + '\n'
            f'Edit V20_CHECKPOINT_DIR / V20_EXP_NAME in cell 4 to point to the directory with your trained v20 RetNet_bestHP fold checkpoints.'
        )

    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    # Prefer the early-stopping-selected BEST weights over the LAST-EPOCH weights.
    # v20-Colab CheckpointManager._epoch_ckpt_path saves both:
    #   'best_model_state_dict' = highest val-QWK weights (this is what the
    #                             reported PANDA QWK 0.9543 was computed from)
    #   'model_state_dict'      = last-completed-epoch weights (for crash resume)
    # Using last-epoch weights would give a biased external-validation estimate.
    if isinstance(checkpoint, dict):
        state = (checkpoint.get('best_model_state_dict')
                 or checkpoint.get('model_state_dict')
                 or checkpoint.get('state_dict')
                 or checkpoint.get('model')
                 or checkpoint)
    else:
        state = checkpoint

    model = make_retnet_besthp_model(MODEL_CONFIG).to(device).eval()
    try:
        model.load_state_dict(state, strict=True)
        print(f'fold {fold}: loaded {os.path.basename(ckpt_path)} (strict)')
    except RuntimeError:
        result = model.load_state_dict(state, strict=False)
        print(f'fold {fold}: loaded {os.path.basename(ckpt_path)} '
              f'(non-strict; missing={len(result.missing_keys)}, unexpected={len(result.unexpected_keys)})')
        if result.missing_keys:
            print(f'  missing keys (first 5): {result.missing_keys[:5]}')
        if result.unexpected_keys:
            print(f'  unexpected keys (first 5): {result.unexpected_keys[:5]}')
    fold_models[fold] = model

print(f'\nLoaded {len(fold_models)}/5 fold checkpoints.\n')

# ============================================================
# Pre-copy TCGA-PRAD features Drive -> local SSD
# ------------------------------------------------------------
# Reading 426 small .pt files directly from Drive triggers
# "OSError: [Errno 107] Transport endpoint is not connected"
# under FUSE mid-loop AND is slow (~8 s per file). One big
# sequential copy avoids both problems.
# ============================================================
import shutil
LOCAL_TCGA_FEATURES_DIR = '/content/local_tcga_features_20x/'
os.makedirs(LOCAL_TCGA_FEATURES_DIR, exist_ok=True)
labels_df    = pd.read_csv(TCGA_LABELS_CSV)
label_lookup = dict(zip(labels_df['slide_id'].astype(str),
                        labels_df['isup_grade'].astype(int)))
drive_files  = sorted([f for f in os.listdir(TCGA_FEATURES_DIR) if f.endswith('.pt')])
print(f'Pre-copying {len(drive_files)} feature files: Drive -> local SSD...')
copied, cached = 0, 0
for fname in tqdm(drive_files, desc='Drive -> local'):
    src = os.path.join(TCGA_FEATURES_DIR, fname)
    dst = os.path.join(LOCAL_TCGA_FEATURES_DIR, fname)
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        cached += 1
        continue
    shutil.copy2(src, dst)
    copied += 1
print(f'  Copied: {copied}, cached (already local): {cached}')

# ============================================================
# Inference (5-fold softmax averaging) — read from LOCAL SSD
# ------------------------------------------------------------
# Resume-safe: skips slides already in predictions.csv.
# Saves predictions.csv every CHECKPOINT_EVERY slides so a
# future Drive disconnect / Colab disconnect loses at most
# CHECKPOINT_EVERY - 1 slides.
# Per-slide try/except so one corrupt .pt does not kill the run.
# ============================================================
feature_files = sorted([f for f in os.listdir(LOCAL_TCGA_FEATURES_DIR) if f.endswith('.pt')])
print(f'TCGA-PRAD features (local): {len(feature_files)} slides')

already_done = set()
predictions = []
if os.path.exists(TCGA_PREDICTIONS_CSV):
    try:
        prev_df = pd.read_csv(TCGA_PREDICTIONS_CSV)
        already_done = set(prev_df['slide_id'].astype(str))
        predictions = prev_df.to_dict('records')
        print(f'Resuming: {len(already_done)} predictions already saved on Drive')
    except Exception as e:
        print(f'(Could not parse existing predictions.csv: {e}; starting fresh)')

CHECKPOINT_EVERY = 20
processed = 0
errors    = []
for fname in tqdm(feature_files, desc='Inference'):
    slide_id = fname.replace('.pt', '')
    if slide_id in already_done or slide_id not in label_lookup:
        continue
    true_isup = label_lookup[slide_id]
    try:
        data = torch.load(os.path.join(LOCAL_TCGA_FEATURES_DIR, fname),
                          weights_only=False)
        features = data['features'].to(device)
        coords   = data.get('coords', None)

        probs_list = []
        with torch.inference_mode():
            for fold, model in fold_models.items():
                logits = model(features, coords,
                               temperature=MODEL_CONFIG['attention_temperature'])
                probs  = F.softmax(logits, dim=-1).cpu().numpy()
                probs_list.append(probs)
        probs_avg = np.mean(probs_list, axis=0)
        pred_isup = int(np.argmax(probs_avg))

        predictions.append({
            'slide_id':   slide_id,
            'true_isup':  true_isup,
            'pred_isup':  pred_isup,
            **{f'prob_isup_{i}': float(probs_avg[i])
               for i in range(MODEL_CONFIG['n_classes'])},
            'top1_confidence': float(probs_avg.max()),
        })
        processed += 1

        # Incremental save to Drive (best-effort; do not crash on save failure)
        if processed % CHECKPOINT_EVERY == 0:
            try:
                pd.DataFrame(predictions).to_csv(TCGA_PREDICTIONS_CSV, index=False)
            except Exception as save_err:
                print(f'\n  WARN: incremental save failed ({save_err}); continuing')
    except Exception as e:
        errors.append((slide_id, str(e)))
        print(f'\n  ERROR {slide_id}: {e}')

predictions_df = pd.DataFrame(predictions)
predictions_df.to_csv(TCGA_PREDICTIONS_CSV, index=False)
print(f'\nPredictions: {len(predictions_df)} slides this session (errors: {len(errors)})')
if errors:
    print(f'First 5 errors:')
    for sid, err in errors[:5]:
        print(f'  - {sid}: {err}')
print(f'Saved: {TCGA_PREDICTIONS_CSV}')


## 9. (SKIPPED) Cancer-specific Evaluation

Benign-side evaluation happens in **Stage 3 locally** via `.claude/bin/audit_dspa_real_benign.py`, which loads both `predictions_20x.csv` (cancer) and `predictions_20x_benign.csv` (benign) and reports protocol-comparable QWK vs DSPA-MIL (Hao et al. MICCAI 2025).

After Cell 8 finishes, `predictions_20x_benign.csv` will be on Drive at `/content/drive/MyDrive/CLAM_TR_Results/TCGA_PRAD/`. Hand off to local Claude session.

In [ ]:
print('Cell 9 SKIPPED. Run Stage 3 locally: python .claude/bin/audit_dspa_real_benign.py')
